## Patient Journey - Lab Test
#### Data source used
* PHCDW.PHCDW_SANDBOX.IQV_QUEST_LAB
* PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PRODUCT 

#### Steps:
1. Query UACR/eGFR lab test claims data after 2017-01-01
2. Get date level lab test stage records (forward fill for between two tests) for each patient
3. Save table to databricks

#### Notes:
1. Time frame: SVC_DT >= '2017-01-01' 
2. GFR test:  upper(LCL_LAB_TEST_NM) like '%GFR%' or upper(LCL_LAB_TEST_NM) like '%GLOMERULAR%' then 'GFR' 
3. UACR test:  upper(LCL_LAB_TEST_NM) like '%ALB%' or upper(LCL_LAB_TEST_NM) like '%RATIO%' then 'UACR'
4. GFR stage: 
    * when test_type = 'GFR'  and MEDIAN_TEST_RESULT <15 then 5
    * when test_type = 'GFR'  and MEDIAN_TEST_RESULT >=15 AND MEDIAN_TEST_RESULT <30 then 4
    * when test_type = 'GFR'  and MEDIAN_TEST_RESULT >=30 AND MEDIAN_TEST_RESULT <60 then 3
    * when test_type = 'GFR'  and MEDIAN_TEST_RESULT >=60 AND MEDIAN_TEST_RESULT <90 then 2
    * when test_type = 'GFR'  and MEDIAN_TEST_RESULT >=90 AND  MEDIAN_TEST_RESULT <=150 then 1
5. UACR stage:
    * when test_type = 'UACR' and MEDIAN_TEST_RESULT <30 then 1
    * when test_type = 'UACR' and MEDIAN_TEST_RESULT >=30 and MEDIAN_TEST_RESULT <300 then 2
    * when test_type = 'UACR' and MEDIAN_TEST_RESULT >=300 then 3
6. Check the the end_date of the data, now we set end_date_limit = Max(SVC_DT)

## 1. Setup Snowflake connection and load data

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

### Load Libraries

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run "./00_cohort_config"

### Get Data from Scope

[REDACTED]

[REDACTED]

[REDACTED]

[REDACTED]



# Cohort Configuration

Shared config notebook for the 6 patient journey validation notebooks. 
Defines the offset, data boundaries, cohort labels (C1–C22), and the cohort assignment function.

**Usage:** Each patient journey notebook should `%run` this notebook early on.

### Define Snowflake Connection Function
Choose appropiate below items based on the Brand
- Role
- Warehouse
- Schema
- Database

OFFSET_DAY      = 0
DATA_START_DATE = 2023-01-01
COHORT_ORIGIN   = 2025-01-01
COHORT_LAST     = 2026-10-01


In [0]:
df_lab = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_0_lab_query_output")

df_lab = df_lab.withColumnRenamed("DATE","SVC_DT")

# Filter to data >= DATA_START_DATE (from config)
df_lab = df_lab.filter(F.col("SVC_DT") >= DATA_START_DATE)

# Validation filter: 10 test HCPs (from config)
df_lab, test_patients = apply_hcp_filter(df_lab)

df_lab = df_lab.select("PATIENT_ID", "SVC_DT", "GFR_STAGE", "UACR_STAGE")

df_lab = df_lab.withColumn("PATIENT_ID", col("PATIENT_ID").cast('string'))

df_lab.display()

Filtered to 244 patients across 10 HCPs


PATIENT_ID,SVC_DT,GFR_STAGE,UACR_STAGE
1745363486,2025-06-04,4,null
789187447,2026-06-17,4,null
788121394,2024-09-02,4,null
302110572,2024-06-18,4,null
302110572,2024-09-07,5,null
809406943,2026-06-24,null,1
941389073,2023-09-15,1,null
713955415,2024-02-13,3,null
2344145081,2023-04-14,2,null
8846290184,2024-08-12,1,null


cohort_label,cohort_ref_month,cohort_start,cohort_end
C1,2025-01-01,2025-01-01,2025-02-01
C2,2025-02-01,2025-02-01,2025-03-01
C3,2025-03-01,2025-03-01,2025-04-01
C4,2025-04-01,2025-04-01,2025-05-01
C5,2025-05-01,2025-05-01,2025-06-01
C6,2025-06-01,2025-06-01,2025-07-01
C7,2025-07-01,2025-07-01,2025-08-01
C8,2025-08-01,2025-08-01,2025-09-01
C9,2025-09-01,2025-09-01,2025-10-01
C10,2025-10-01,2025-10-01,2025-11-01



22 cohorts defined (C1 – C22)


Testing with current OFFSET_DAY = 0
-------------------------------------------------------
  2025-01-01 → 2025-01-01  (expected 2025-01-01)  [PASS]

Config notebook ready. All 6 patient journey notebooks can %run this.


## 2. Data cleanup and processing

In [0]:
# Cohort-aware month-grain cleanup with forward-fill

import sys
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.functions import col

# Find the end date limit
end_date_limit = str(df_lab.agg({"SVC_DT": "max"}).collect()[0]['max(SVC_DT)'])

# Define forward fill windows (cohort-month-based)
window_month = Window.partitionBy('PATIENT_ID').orderBy('COHORT_MONTH').rowsBetween(-sys.maxsize, 0)

# Get LAST_TEST_DATE per patient
df_lab_with_last_date = df_lab.withColumn(
    "LAST_TEST_DATE", 
    F.lead("SVC_DT", 1).over(Window.partitionBy("PATIENT_ID").orderBy("SVC_DT"))
).withColumn("LAST_TEST_DATE", F.coalesce(col("LAST_TEST_DATE"), F.to_date(F.lit(end_date_limit))))

# Generate cohort-month sequence from SVC_DT to LAST_TEST_DATE
# Uses assign_cohort_month() from config to shift by offset
df_lab_with_month_seq = df_lab_with_last_date.withColumn(
    "MONTH_SEQ",
    F.expr(f"""
        sequence(
            {_cohort_month_sql('SVC_DT')},
            {_cohort_month_sql('LAST_TEST_DATE')},
            interval 1 month
        )
    """)
)

# Explode the month sequence to get each cohort-month in the range
df_lab_expanded = df_lab_with_month_seq.withColumn(
    "COHORT_MONTH",
    F.explode("MONTH_SEQ")
).drop("MONTH_SEQ")

# Forward-fill test stages across cohort-months
filled_gfr = F.last(col("GFR_STAGE"), ignorenulls=True).over(window_month)
filled_uacr = F.last(col("UACR_STAGE"), ignorenulls=True).over(window_month)

df_lab_filled = df_lab_expanded.withColumn("GFR_STAGE", filled_gfr).withColumn("UACR_STAGE", filled_uacr)

# Aggregate to remove duplicates per patient-cohort-month and take the max stages
df_lab_clean = df_lab_filled.groupBy("PATIENT_ID", "COHORT_MONTH").agg(
    F.max("GFR_STAGE").alias("GFR_STAGE"),
    F.max("UACR_STAGE").alias("UACR_STAGE")
).orderBy("PATIENT_ID", "COHORT_MONTH")

# Join cohort labels (C1-C22); rows before C1 get null label (lookback)
df_lab_clean = df_lab_clean.join(
    df_cohort_ref.select("cohort_ref_month", "cohort_label"),
    df_lab_clean.COHORT_MONTH == df_cohort_ref.cohort_ref_month,
    "left"
).drop("cohort_ref_month")

df_lab_clean.display()

PATIENT_ID,COHORT_MONTH,GFR_STAGE,UACR_STAGE,cohort_label
1015469304,2023-03-01,1,null,null
1015469304,2023-04-01,1,null,null
1015469304,2023-05-01,1,null,null
1015469304,2023-06-01,1,null,null
1015469304,2023-07-01,1,null,null
1015469304,2023-08-01,1,null,null
1015469304,2023-09-01,1,null,null
1015469304,2023-10-01,1,null,null
1015469304,2023-11-01,1,null,null
1015469304,2023-12-01,1,null,null


In [0]:
df_lab_clean = df_lab_clean.withColumnRenamed("COHORT_MONTH", "DATE")

In [0]:
df_lab_clean.count(), len(df_lab_clean.columns)

(6911, 5)

In [0]:
# ============================================================
# COMPARISON: offset=0 validation vs production (same patients)
# ============================================================
base = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output"
prod_lab = spark.read.format("delta").load(f"{base}/kerendia_nba_3_0_lab_journey_output")
val_patients = [r.PATIENT_ID for r in df_lab_clean.select("PATIENT_ID").distinct().collect()]
prod_lab_f = prod_lab.filter(col("PATIENT_ID").isin(val_patients))

val_count = df_lab_clean.count()
prod_count = prod_lab_f.count()
print(f"Validation (offset=0): {val_count:,}")
print(f"Production (same pts): {prod_count:,}")
print(f"Difference:            {val_count - prod_count:+,}")

# Key comparisons
val_keys = df_lab_clean.select("PATIENT_ID", "DATE")
prod_keys = prod_lab_f.select("PATIENT_ID", "DATE")

only_in_val = val_keys.subtract(prod_keys)
only_in_prod = prod_keys.subtract(val_keys)
print(f"\nRows only in validation: {only_in_val.count()}")
print(f"Rows only in production: {only_in_prod.count()}")

# Attribute: how many prod-only rows are before DATA_START_DATE?
prod_only_before = only_in_prod.filter(col("DATE") < DATA_START_DATE).count()
prod_only_after = only_in_prod.filter(col("DATE") >= DATA_START_DATE).count()
print(f"  Prod-only before {DATA_START_DATE}: {prod_only_before} (DATA_START_DATE filter)")
print(f"  Prod-only on/after {DATA_START_DATE}: {prod_only_after}")

# Stage differences on overlapping rows
common = df_lab_clean.alias("v").join(
    prod_lab_f.alias("p"),
    (col("v.PATIENT_ID") == col("p.PATIENT_ID")) & (col("v.DATE") == col("p.DATE")),
    "inner"
)
stage_diffs = common.filter(
    (col("v.GFR_STAGE") != col("p.GFR_STAGE")) |
    (col("v.UACR_STAGE") != col("p.UACR_STAGE")) |
    (col("v.GFR_STAGE").isNull() != col("p.GFR_STAGE").isNull()) |
    (col("v.UACR_STAGE").isNull() != col("p.UACR_STAGE").isNull())
)
print(f"\nOverlapping rows: {common.count():,}")
print(f"Stage value differences: {stage_diffs.count()}")
if stage_diffs.count() > 0:
    print("\nSample diffs:")
    stage_diffs.select(
        col("v.PATIENT_ID"), col("v.DATE"),
        col("v.GFR_STAGE").alias("val_GFR"), col("p.GFR_STAGE").alias("prod_GFR"),
        col("v.UACR_STAGE").alias("val_UACR"), col("p.UACR_STAGE").alias("prod_UACR")
    ).orderBy("PATIENT_ID", "DATE").show(20, truncate=False)

Validation (offset=0): 6,911
Production (same pts): 7,985
Difference:            -1,074

Rows only in validation: 0
Rows only in production: 1074
  Prod-only before 2023-01-01: 703 (DATA_START_DATE filter)
  Prod-only on/after 2023-01-01: 371

Overlapping rows: 6,911
Stage value differences: 88

Sample diffs:
+----------+----------+-------+--------+--------+---------+
|PATIENT_ID|DATE      |val_GFR|prod_GFR|val_UACR|prod_UACR|
+----------+----------+-------+--------+--------+---------+
|1015469304|2023-03-01|1      |2       |NULL    |NULL     |
|130257011 |2023-02-01|1      |2       |NULL    |NULL     |
|1353683025|2023-03-01|NULL   |2       |1       |1        |
|1353683025|2023-04-01|NULL   |2       |1       |1        |
|1353683025|2023-05-01|NULL   |2       |1       |1        |
|1422311670|2023-02-01|2      |2       |NULL    |1        |
|1422311670|2023-03-01|2      |2       |NULL    |1        |
|1422311670|2023-04-01|2      |2       |NULL    |1        |
|1422311670|2023-05-01|2     

In [0]:
df_lab.count(), len(df_lab.columns)

(1355, 4)

In [0]:


# VALIDATION: write to a separate path so we don't overwrite production data
df_lab_clean.write.format("delta").mode("overwrite").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_lab_journey_validation")